<a href="https://colab.research.google.com/github/avijitdeb2023/Data-Science/blob/main/Homework%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Question 4(b): Damped Newton Method with Backtracking**

In [2]:
import pandas as pd
from IPython.display import display


def f(x):
    return 2 * x**2 + x - 4


def df(x):
    return 4 * x + 1
x = 0.0
tolerance = 1e-8
max_iterations = 100
results = []

for n in range(max_iterations):
    fx = f(x)

    if fx == 0:
        print("An exact root has been reached.")
        break

    if df(x) == 0:
        raise ValueError("The derivative is zero; cannot compute the direction.")
    d = -fx / df(x)
    lam = 1.0
    for _ in range(100):
        x_new = x + lam * d

        if abs(f(x_new)) < abs(fx):
            break

        lam /= 2
    else:
        raise RuntimeError("Backtracking failed to find an acceptable step.")
    step = abs(x_new - x)

    results.append({
        "n": n,
        "x_n": x,
        "d_n": d,
        "lambda_n": lam,
        "|f(x_n)|": abs(fx),
        "x_(n+1)": x_new,
        "|x_(n+1) - x_n|": step
    })

    x = x_new
    if step < tolerance:
        break
else:
    raise RuntimeError("Maximum number of iterations exceeded.")
table = pd.DataFrame(results)

display(
    table.style
    .hide(axis="index")
    .format({
        "n": "{:.0f}",
        "x_n": "{:.15f}",
        "d_n": "{:.12e}",
        "lambda_n": "{:.2f}",
        "|f(x_n)|": "{:.12e}",
        "x_(n+1)": "{:.15f}",
        "|x_(n+1) - x_n|": "{:.12e}"
    })
)

print(f"Approximate root: {x:.15f}")
print(f"Number of accepted updates: {len(results)}")
print(f"Final step size: {step:.12e}")
print(f"Final residual: {abs(f(x)):.12e}")

print(
    "\nComment: The residual decreases at every accepted step. "
    "The first update uses lambda = 0.25; all later updates "
    "accept the full Newton step, lambda = 1."
)

n,x_n,d_n,lambda_n,|f(x_n)|,x_(n+1),|x_(n+1) - x_n|
0,0.000000000000000,4.000000000000e+00,0.25,4.000000000000e+00,1.000000000000000,1.000000000000e+00
1,1.000000000000000,2.000000000000e-01,1.00,1.000000000000e+00,1.200000000000000,2.000000000000e-01
2,1.200000000000000,-1.379310344828e-02,1.00,8.000000000000e-02,1.186206896551724,1.379310344828e-02
3,1.186206896551724,-6.623338990769e-05,1.00,3.804994054697e-04,1.186140663161816,6.623338990774e-05
4,1.186140663161816,-1.527309146389e-09,1.00,8.773723081390e-09,1.186140661634507,1.527309168026e-09


Approximate root: 1.186140661634507
Number of accepted updates: 5
Final step size: 1.527309168026e-09
Final residual: 0.000000000000e+00

Comment: The residual decreases at every accepted step. The first update uses lambda = 0.25; all later updates accept the full Newton step, lambda = 1.


# **Question 5: Comparison of Iterative Methods**

In [3]:
import pandas as pd
from IPython.display import display


# Functions used in the methods
def f(x):
    return x**3 + x - 1


def df(x):
    return 3 * x**2 + 1


def g(x):
    return 1 / (x**2 + 1)


tolerance = 1e-8
max_iterations = 100
summary = []


# Display a neatly formatted iteration table
def show_results(title, rows, root, step):
    print(f"\n{title}")

    table = pd.DataFrame(rows)
    formatting = {
        column: "{:.15f}"
        for column in table.columns
        if column != "n"
    }
    formatting["n"] = "{:.0f}"
    formatting["|f(x_(n+1))|"] = "{:.12e}"

    display(
        table.style
        .hide(axis="index")
        .format(formatting)
    )

    print(f"Approximate root: {root:.15f}")
    print(f"Number of updates: {len(rows)}")
    print(f"Final step size: {step:.12e}")

    summary.append({
        "Method": title,
        "Updates": len(rows),
        "Approximate root": root,
        "Final step size": step
    })


# =========================================================
# (a) Steffensen's Method: x0 = 0.5
# =========================================================
x = 0.5
rows = []

for n in range(max_iterations):
    y = g(x)
    z = g(y)
    denominator = z - 2 * y + x

    if denominator == 0:
        raise ValueError("Steffensen: denominator is zero.")

    x_new = x - (y - x)**2 / denominator
    step = abs(x_new - x)

    rows.append({
        "n": n,
        "x_n": x,
        "x_(n+1)": x_new,
        "|f(x_(n+1))|": abs(f(x_new))
    })

    x = x_new

    if step < tolerance:
        break
else:
    raise RuntimeError("Steffensen: maximum iterations exceeded.")

show_results("Steffensen's Method", rows, x, step)


# =========================================================
# (b) Standard Newton's Method: x0 = 0.5
# =========================================================
x = 0.5
rows = []

for n in range(max_iterations):
    derivative = df(x)

    if derivative == 0:
        raise ValueError("Newton: derivative is zero.")

    x_new = x - f(x) / derivative
    step = abs(x_new - x)

    rows.append({
        "n": n,
        "x_n": x,
        "x_(n+1)": x_new,
        "|f(x_(n+1))|": abs(f(x_new))
    })

    x = x_new

    if step < tolerance:
        break
else:
    raise RuntimeError("Newton: maximum iterations exceeded.")

show_results("Newton's Method", rows, x, step)


# =========================================================
# (c) Secant Method: x0 = 0.5, x1 = 1.0
# =========================================================
x_prev = 0.5
x = 1.0
f_prev = f(x_prev)
fx = f(x)
rows = []

for n in range(1, max_iterations + 1):
    denominator = fx - f_prev

    if denominator == 0:
        raise ValueError("Secant: denominator is zero.")

    x_new = x - fx * (x - x_prev) / denominator
    f_new = f(x_new)
    step = abs(x_new - x)

    rows.append({
        "n": n,
        "x_(n-1)": x_prev,
        "x_n": x,
        "x_(n+1)": x_new,
        "|f(x_(n+1))|": abs(f_new)
    })

    # Update the two points and their function values.
    x_prev, x = x, x_new
    f_prev, fx = fx, f_new

    if step < tolerance:
        break
else:
    raise RuntimeError("Secant: maximum iterations exceeded.")

show_results("Secant Method", rows, x, step)


# =========================================================
# Final comparison
# =========================================================
print("\nCOMPARISON OF THE THREE METHODS")

display(
    pd.DataFrame(summary).style
    .hide(axis="index")
    .format({
        "Updates": "{:.0f}",
        "Approximate root": "{:.15f}",
        "Final step size": "{:.12e}"
    })
)


Steffensen's Method


n,x_n,x_(n+1),|f(x_(n+1))|
0,0.500000000000000,0.683582089552239,3.009386127948e-03
1,0.683582089552239,0.682327880927780,1.847860646720e-07
2,0.682327880927780,0.682327803828020,6.661338147751e-16
3,0.682327803828020,0.682327803828019,2.220446049250e-16


Approximate root: 0.682327803828019
Number of updates: 4
Final step size: 2.220446049250e-16

Newton's Method


n,x_n,x_(n+1),|f(x_(n+1))|
0,0.500000000000000,0.714285714285714,7.871720116618e-02
1,0.714285714285714,0.683179723502304,2.043293802509e-03
2,0.683179723502304,0.682328423304578,1.484708738619e-06
3,0.682328423304578,0.682327803828347,7.855938122248e-13
4,0.682327803828347,0.682327803828019,2.220446049250e-16


Approximate root: 0.682327803828019
Number of updates: 5
Final step size: 3.277378368693e-13

Secant Method


n,x_(n-1),x_n,x_(n+1),|f(x_(n+1))|
1,0.500000000000000,1.000000000000000,0.636363636363636,1.059353869271e-01
2,1.000000000000000,0.636363636363636,0.671195652173913,2.642828787011e-02
3,0.636363636363636,0.671195652173913,0.682773870368871,1.069501175636e-03
4,0.671195652173913,0.682773870368871,0.682323546237331,1.020418880637e-05
5,0.682773870368871,0.682323546237331,0.682327802206241,3.886939103914e-09
6,0.682323546237331,0.682327802206241,0.682327803828025,1.421085471520e-14


Approximate root: 0.682327803828025
Number of updates: 6
Final step size: 1.621784595507e-09

COMPARISON OF THE THREE METHODS


Method,Updates,Approximate root,Final step size
Steffensen's Method,4,0.682327803828019,2.220446049250e-16
Newton's Method,5,0.682327803828019,3.277378368693e-13
Secant Method,6,0.682327803828025,1.621784595507e-09
